In [ ]:

from dotenv import load_dotenv
import os
load_dotenv()
from openai import OpenAI
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
client = OpenAI()

import json
from pathlib import Path
from openai import OpenAI

client = OpenAI()

# Pfade
calls_dir = Path("../../emergency_calls/calls_german").resolve()
guidelines_dir = Path("guidelines").resolve()

assert calls_dir.exists(), f"calls_dir nicht gefunden: {calls_dir}"
assert guidelines_dir.exists(), f"guidelines_dir nicht gefunden: {guidelines_dir}"

# Notrufe sammeln
call_files = sorted(list(calls_dir.glob("*.md")) + list(calls_dir.glob("*.txt")))
assert call_files, f"Keine Notrufe gefunden in {calls_dir}"

# Guidelines je Domäne laden
DOMAIN_FILES = {
    "A": guidelines_dir / "a_guidelines.md",
    "B": guidelines_dir / "b_guidelines.md",
    "C": guidelines_dir / "c_guidelines.md",
    "D": guidelines_dir / "d_guidelines.md",
    "E": guidelines_dir / "e_guidelines.md",
}
domain_guidelines = {}
for d, p in DOMAIN_FILES.items():
    assert p.exists(), f"Guideline fehlt: {p}"
    domain_guidelines[d] = p.read_text(encoding="utf-8").strip()

# Output Contract (minimal, pro Domain)
output_contract = """
Return ONLY valid JSON (no markdown, no commentary).

Schema:
{
  "severity": <int or null>,
  "confidence": <float 0..1 or null>,
  "findings": <list of strings>
}

Rules:
- findings must be short factual phrases grounded in the call text.
- if no findings are available, return [].
- do NOT invent findings.
"""

MODEL = "gpt-5.2"
TEMPERATURE = 1

batch_input_path = Path("batch_input_abcde.jsonl").resolve()

with batch_input_path.open("w", encoding="utf-8") as f:

    for fp in call_files:

        transcript = fp.read_text(encoding="utf-8").strip()

        for domain in ["A", "B", "C", "D", "E"]:
            req = {
            "custom_id": f"{fp.stem}__{domain}",
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": MODEL,
                "temperature": TEMPERATURE,
                "input": [
                    {"role": "system", "content": [{"type": "input_text", "text": domain_guidelines[domain]}]},
                    {"role": "system", "content": [{"type": "input_text", "text": output_contract}]},
                    {"role": "user",   "content": [{"type": "input_text", "text": transcript}]},
                ],
                # JSON erzwingen (wie zuvor)
                "text": {"format": {"type": "json_object"}},
            },
        }
            f.write(json.dumps(req, ensure_ascii=False) + "\n")


print("Geschrieben:", batch_input_path)
print("Notrufe:", len(call_files), "=> Requests:", len(call_files) * 5)

In [ ]:
upload = client.files.create(file=batch_input_path.open("rb"), purpose="batch")

batch = client.batches.create(
    input_file_id=upload.id,
    endpoint="/v1/responses",
    completion_window="24h"
)

print("batch_id:", batch.id, "status:", batch.status)

In [ ]:

b = client.batches.retrieve(batch.id)
print("status:", b.status)
print("output_file_id:", b.output_file_id)
print("error_file_id:", b.error_file_id)

In [ ]:
# Output herunterladen (wenn status == "completed")
out_path = Path("batch_output.jsonl").resolve()

b = client.batches.retrieve(batch.id)
assert b.output_file_id, "Noch kein output_file_id vorhanden (Batch evtl. noch nicht fertig)."

content = client.files.content(b.output_file_id)
out_path.write_bytes(content.read())

print("Gespeichert:", out_path)

In [ ]:
import json
import re
import pandas as pd
from pathlib import Path

def _extract_json_from_response_obj(resp_body: dict) -> dict:
    out = resp_body.get("output", [])
    texts = []
    for item in out:
        for c in item.get("content", []):
            if c.get("type") in ("output_text", "text") and "text" in c:
                texts.append(c["text"])
    if not texts and resp_body.get("output_text"):
        texts = [resp_body["output_text"]]
    if not texts:
        raise ValueError("Kein output_text gefunden")

    text = "\n".join(texts).strip()
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        raise ValueError(f"Kein JSON im Text: {text[:200]}...")
    return json.loads(m.group(0))

def parse_batch_output_to_df(batch_output_jsonl: str) -> pd.DataFrame:
    # Sammelstruktur: {Notruf_12: {"severity_A":..., ...}}
    by_call = {}

    lines = Path(batch_output_jsonl).read_text(encoding="utf-8").splitlines()
    for line in lines:
        if not line.strip():
            continue
        rec = json.loads(line)

        custom_id = rec.get("custom_id")  # Notruf_12__A
        if not custom_id or "__" not in custom_id:
            continue

        call_id, domain = custom_id.split("__", 1)
        domain = domain.upper()

        by_call.setdefault(call_id, {"file": call_id})

        err = rec.get("error")
        if err:
            by_call[call_id][f"severity_{domain}"] = None
            by_call[call_id][f"confidence_{domain}"] = None
            by_call[call_id][f"findings_{domain}"] = json.dumps([], ensure_ascii=False)
            by_call[call_id][f"error_{domain}"] = json.dumps(err, ensure_ascii=False)
            continue

        body = (rec.get("response") or {}).get("body") or {}
        try:
            data = _extract_json_from_response_obj(body)
            sev = data.get("severity")
            conf = data.get("confidence")
            findings = data.get("findings", [])
            if findings is None:
                findings = []
            if not isinstance(findings, list):
                findings = [str(findings)]
        except Exception as e:
            by_call[call_id][f"severity_{domain}"] = None
            by_call[call_id][f"confidence_{domain}"] = None
            by_call[call_id][f"findings_{domain}"] = json.dumps([], ensure_ascii=False)
            by_call[call_id][f"error_{domain}"] = f"parse_error: {e}"
            continue

        by_call[call_id][f"severity_{domain}"] = sev
        by_call[call_id][f"confidence_{domain}"] = conf
        by_call[call_id][f"findings_{domain}"] = json.dumps(findings, ensure_ascii=False)
        by_call[call_id][f"error_{domain}"] = None

    df = pd.DataFrame(list(by_call.values()))

    # call_nr für Sortierung
    df["call_nr"] = df["file"].str.extract(r"(\d+)").astype("Int64")
    df = df.sort_values(["call_nr", "file"], na_position="last").reset_index(drop=True)

    # Spaltenreihenfolge hübsch
    cols = ["file", "call_nr"]
    for d in ["A","B","C","D","E"]:
        cols += [f"severity_{d}", f"confidence_{d}", f"findings_{d}", f"error_{d}"]
    for c in cols:
        if c not in df.columns:
            df[c] = None
    df = df[cols]

    return df

df = parse_batch_output_to_df("batch_output.jsonl")
df.head()

In [ ]:
out_csv = Path("run_3.csv").resolve()
df.to_csv(out_csv, index=False, encoding="utf-8")
print("Gespeichert:", out_csv)

In [ ]:
df